# Memory-Augmented Chatbot with Knowledge Graph and Hybrid RAG System — Gemini Edition

This notebook is a **complete implementation-oriented prototype** of the supplied project specification.

It covers:
- Web scraping → cleaning → chunking
- Embeddings + FAISS vector retrieval
- Entity/relationship extraction + Knowledge Graph
- **Hybrid retrieval:** semantic vector search + graph facts
- Persistent user memory with MongoDB support and JSON fallback
- Dynamic live-web tool
- LangGraph orchestration and routing
- LLM-based evaluation: context relevance, faithfulness, answer correctness
- FastAPI `/chat` endpoint for serving the chatbot

> **Important:** Neo4j and MongoDB connections are enabled automatically when their environment variables are configured. Without credentials, the notebook remains runnable using NetworkX + JSON fallback.


## 0. Setup — Install Dependencies

In [ ]:
# Core LLM orchestration
!pip -q install -U langgraph langchain langchain-core langchain-google-genai langchain-community

# Embeddings + vector store
!pip -q install sentence-transformers faiss-cpu

# Knowledge graph + visualization
!pip -q install networkx pyvis neo4j

# Entity/relationship extraction
!pip -q install spacy
!python -m spacy download en_core_web_sm -q

# Web scraping
!pip -q install beautifulsoup4 requests lxml

# Dynamic/real-time tool
!pip -q install duckduckgo-search

# Persistent memory (MongoDB) + API layer (FastAPI)
!pip -q install pymongo fastapi uvicorn nest-asyncio

print("All dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 61.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
All dependencies installed.


In [ ]:
import os
import json
import time
import getpass
import requests
import numpy as np
import networkx as nx
from bs4 import BeautifulSoup
from typing import TypedDict, List, Dict, Any, Optional

# ---- GOOGLE GEMINI API KEY / CONNECTIONS ---------------------------------
# This project uses Google Gemini for the LLM.
# In Google Colab, add a secret named GOOGLE_API_KEY.
# If the secret is unavailable, you may enter your Gemini API key interactively.

if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
    except Exception:
        pass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass(
        "Enter your Google Gemini API key: "
    )

# Optional production connections. Leave blank to use local fallbacks.
NEO4J_URI = os.getenv("NEO4J_URI", "")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "")
MONGODB_URI = os.getenv("MONGODB_URI", "")
MONGODB_DB = os.getenv("MONGODB_DB", "chatbot_db")

print("Environment ready.")
print("Gemini API key configured:", bool(os.environ.get("GOOGLE_API_KEY")))
print("Neo4j configured:", bool(NEO4J_URI and NEO4J_PASSWORD))
print("MongoDB configured:", bool(MONGODB_URI))


Environment ready.
Gemini API key configured: True
Neo4j configured: False
MongoDB configured: False


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

def get_llm(temperature: float = 0.2):
    """Gemini LLM used throughout the notebook."""
    return ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        temperature=temperature,
        google_api_key=os.environ["GOOGLE_API_KEY"],
    )

def get_embeddings():
    """Google Gemini embedding model used for the vector store."""
    return GoogleGenerativeAIEmbeddings(
        model="models/text-embedding-004",
        google_api_key=os.environ["GOOGLE_API_KEY"],
    )

llm = get_llm()
embeddings = get_embeddings()
print("Gemini LLM and embedding clients initialized.")


Gemini LLM and embedding clients initialized.


## 1. Data Pipeline (Static Knowledge Layer)

**Step 1 of the methodology:** web scraping → cleaning → chunking.

Replace `SOURCE_URLS` with the pages you want your chatbot to know about (docs, wiki pages,
blog posts, company FAQs, etc.).


In [ ]:
SOURCE_URLS = [
    "https://en.wikipedia.org/wiki/Retrieval-augmented_generation",
    "https://en.wikipedia.org/wiki/Knowledge_graph",
    "https://en.wikipedia.org/wiki/Chatbot",
]

def scrape_page(url: str) -> str:
    """Fetch a page and extract clean visible text."""
    try:
        resp = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"Failed to fetch {url}: {e}")
        return ""

    soup = BeautifulSoup(resp.text, "lxml")

    # Strip non-content elements
    for tag in soup(["script", "style", "nav", "footer", "header", "aside", "table"]):
        tag.decompose()

    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    text = "\n".join(p for p in paragraphs if len(p) > 40)
    return text

def clean_text(text: str) -> str:
    """Basic normalization/cleaning."""
    text = text.replace("\xa0", " ")
    text = " ".join(text.split())  # collapse whitespace
    return text

raw_documents = []
for url in SOURCE_URLS:
    print(f"Scraping: {url}")
    text = scrape_page(url)
    if text:
        raw_documents.append({"source": url, "text": clean_text(text)})
    time.sleep(1)  # be polite to servers

print(f"\nScraped {len(raw_documents)} documents.")
if raw_documents:
    print("Sample (first 400 chars):\n", raw_documents[0]["text"][:400])


Scraping: https://en.wikipedia.org/wiki/Retrieval-augmented_generation
Scraping: https://en.wikipedia.org/wiki/Knowledge_graph
Scraping: https://en.wikipedia.org/wiki/Chatbot

Scraped 3 documents.
Sample (first 400 chars):
 Retrieval-augmented generation ( RAG ) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. [ 1 ] With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data . [ 2 ] This allows LLMs to use domain-specific and/or 


In [ ]:
def chunk_text(text: str, chunk_size: int = 800, overlap: int = 100) -> List[str]:
    """Simple sliding-window chunker (character-based, dependency-free)."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return [c.strip() for c in chunks if len(c.strip()) > 50]

all_chunks = []  # list of dicts: {"text":..., "source":...}
for doc in raw_documents:
    for chunk in chunk_text(doc["text"]):
        all_chunks.append({"text": chunk, "source": doc["source"]})

print(f"Created {len(all_chunks)} chunks from {len(raw_documents)} documents.")


Created 55 chunks from 3 documents.


## 2. Embedding & Vector Storage (FAISS)

**Step 2 of the methodology:** generate embeddings, store in a vector database. FAISS is used
here (swap for Chroma with `langchain_community.vectorstores.Chroma` if you prefer a persistent
on-disk store).


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

lc_documents = [
    Document(page_content=c["text"], metadata={"source": c["source"]})
    for c in all_chunks
]

vector_store = FAISS.from_documents(lc_documents, embeddings)
print(f"FAISS vector store built with {len(lc_documents)} chunks.")

# Persist to disk so you don't have to re-embed every run
vector_store.save_local("faiss_index")
print("Saved index to ./faiss_index")

# To reload later:
# vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)


/tmp/ipykernel_9032/1573098637.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


GoogleGenerativeAIError: Error embedding content (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

## 3. Knowledge Graph Construction

**Step 3 of the methodology:** entity extraction → relationship mapping → graph storage.

We use spaCy for lightweight entity extraction and co-occurrence-based relationships, stored in
an in-memory **NetworkX** graph (free, no server needed). A commented **Neo4j** block is provided
below if you have a Neo4j Aura free-tier instance and want the "real" graph database.


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")
kg = nx.DiGraph()

ENTITY_LABELS = {"PERSON", "ORG", "GPE", "PRODUCT", "EVENT", "WORK_OF_ART", "NORP", "FAC", "LAW", "TECHNOLOGY"}

def extract_entities_relations(text: str, source: str):
    """Extract entities and map sentence-level relationships into a graph.
    The co-occurrence relation is an explicit, reproducible relationship heuristic.
    """
    doc = nlp(text)
    for sent in doc.sents:
        ents = [ent for ent in sent.ents if ent.label_ in ENTITY_LABELS]
        for ent in ents:
            name = ent.text.strip()
            if name:
                kg.add_node(name, label=ent.label_, source=source)
        for i in range(len(ents)):
            for j in range(i + 1, len(ents)):
                a, b = ents[i].text.strip(), ents[j].text.strip()
                if a and b and a != b:
                    kg.add_edge(a, b, relation="co_occurs_with", source=source)

for doc in raw_documents:
    extract_entities_relations(doc["text"], doc["source"])

print(f"NetworkX graph: {kg.number_of_nodes()} entities, {kg.number_of_edges()} relations.")
print("Sample nodes:", list(kg.nodes)[:10])


In [ ]:
# --- Optional: visualize the knowledge graph inline ---
from pyvis.network import Network

net = Network(notebook=True, cdn_resources="in_line", height="500px", width="100%", directed=True)
# Limit to top-connected nodes so the graph stays readable
top_nodes = sorted(kg.degree, key=lambda x: x[1], reverse=True)[:40]
sub_kg = kg.subgraph([n for n, _ in top_nodes])
net.from_nx(sub_kg)
net.show("knowledge_graph.html")

from IPython.display import HTML
HTML("knowledge_graph.html")


In [ ]:
# Neo4j integration ---------------------------------------------------------
# If NEO4J_URI and NEO4J_PASSWORD are configured, the same extracted graph is
# persisted into Neo4j. Otherwise NetworkX remains the local fallback.

from neo4j import GraphDatabase

neo4j_driver = None
if NEO4J_URI and NEO4J_PASSWORD:
    try:
        neo4j_driver = GraphDatabase.driver(
            NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
        )
        neo4j_driver.verify_connectivity()
        print("Connected to Neo4j.")
    except Exception as e:
        print("Neo4j connection unavailable; using NetworkX fallback:", e)
        neo4j_driver = None

def push_to_neo4j(graph: nx.DiGraph):
    if neo4j_driver is None:
        return False
    with neo4j_driver.session() as session:
        for node, data in graph.nodes(data=True):
            session.run(
                "MERGE (e:Entity {name: $name}) SET e.label = $label, e.source = $source",
                name=node, label=data.get("label", "UNKNOWN"), source=data.get("source", "")
            )
        for u, v, data in graph.edges(data=True):
            session.run(
                """MATCH (a:Entity {name: $u}), (b:Entity {name: $v})
                   MERGE (a)-[r:RELATED {type: $rel}]->(b)
                   SET r.source = $source""",
                u=u, v=v, rel=data.get("relation", "related_to"), source=data.get("source", "")
            )
    return True

if push_to_neo4j(kg):
    print("Knowledge graph persisted to Neo4j.")
else:
    print("Neo4j not configured: NetworkX is the active local graph store.")


In [ ]:
def extract_query_entities(query: str) -> List[str]:
    """Extract entities from the user question for graph retrieval."""
    doc = nlp(query)
    entities = [e.text.strip() for e in doc.ents if e.text.strip()]
    # Also match known graph nodes by phrase overlap.
    q_lower = query.lower()
    for node in kg.nodes:
        if node.lower() in q_lower and node not in entities:
            entities.append(node)
    return list(dict.fromkeys(entities))

def query_knowledge_graph(entity: str, depth: int = 1, limit: int = 12) -> List[str]:
    """Query Neo4j when available, otherwise query the local NetworkX graph."""
    if neo4j_driver is not None:
        cypher = """
        MATCH (e:Entity)
        WHERE toLower(e.name) CONTAINS toLower($entity)
        OPTIONAL MATCH (e)-[r:RELATED]->(n:Entity)
        RETURN e.name AS source, r.type AS relation, n.name AS target
        LIMIT $limit
        """
        with neo4j_driver.session() as session:
            rows = session.run(cypher, entity=entity, limit=limit)
            return [f"{r['source']} --{r['relation']}--> {r['target']}" for r in rows if r['target']]

    facts = []
    matches = [n for n in kg.nodes if entity.lower() in n.lower()]
    for m in matches:
        for neighbor in kg.successors(m):
            rel = kg[m][neighbor].get("relation", "related_to")
            facts.append(f"{m} --{rel}--> {neighbor}")
        for neighbor in kg.predecessors(m):
            rel = kg[neighbor][m].get("relation", "related_to")
            facts.append(f"{neighbor} --{rel}--> {m}")
    return facts[:limit]

def retrieve_graph_facts(query: str) -> List[str]:
    entities = extract_query_entities(query)
    facts = []
    for entity in entities[:5]:
        facts.extend(query_knowledge_graph(entity))
    return list(dict.fromkeys(facts))[:20]

print("Graph retrieval test:", retrieve_graph_facts("What is related to artificial intelligence?"))


## 4. RAG Pipeline

**Step 4 of the methodology:** query embedding → similarity search → context retrieval → LLM
answer generation.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant. Answer using the provided semantic context and structured "
     "knowledge-graph facts. Prefer evidence from the supplied sources. If the evidence does not "
     "support the answer, say you don't know. Do not invent facts."),
    ("human",
     "Semantic vector context:\n{context}\n\n"
     "Structured knowledge-graph facts:\n{kg_facts}\n\n"
     "Question: {question}"),
])

def retrieve_context(query: str, k: int = 4):
    docs = vector_store.similarity_search(query, k=k)
    context = "\n\n".join(
        f"[{d.metadata.get('source')}] {d.page_content}" for d in docs
    )
    return context, docs

def hybrid_retrieve(query: str, k: int = 4) -> Dict[str, Any]:
    """Hybrid retrieval = semantic vector search + structured graph retrieval."""
    context, docs = retrieve_context(query, k=k)
    kg_facts_list = retrieve_graph_facts(query)
    kg_facts = "\n".join(kg_facts_list) if kg_facts_list else "No relevant graph facts found."
    return {
        "context": context,
        "docs": docs,
        "kg_facts": kg_facts,
        "graph_facts": kg_facts_list,
    }

def rag_answer(query: str) -> Dict[str, Any]:
    retrieved = hybrid_retrieve(query)
    chain = RAG_PROMPT | llm
    response = chain.invoke({
        "context": retrieved["context"],
        "kg_facts": retrieved["kg_facts"],
        "question": query,
    })
    return {
        "answer": response.content,
        "context": retrieved["context"],
        "kg_facts": retrieved["kg_facts"],
        "graph_facts": retrieved["graph_facts"],
        "sources": list({d.metadata.get("source") for d in retrieved["docs"]}),
    }

result = rag_answer("What is retrieval-augmented generation?")
print(result["answer"])
print("\nVector sources:", result["sources"])
print("Graph facts:", result["graph_facts"][:5])


## 5. Long-Term Memory

Per-user memory that persists across turns (and across sessions, since it's saved to disk).
Memory is itself embedded so it can be *semantically* retrieved — e.g. "what did I say my
favorite language was?" — not just replayed in order.


In [ ]:
# Persistent memory: MongoDB when configured, JSON fallback for zero-setup Colab.
MEMORY_PATH = "user_memory.json"

from pymongo import MongoClient

mongo_client = None
mongo_collection = None
if MONGODB_URI:
    try:
        mongo_client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5000)
        mongo_client.admin.command("ping")
        mongo_collection = mongo_client[MONGODB_DB]["memory"]
        print("Connected to MongoDB persistent memory.")
    except Exception as e:
        print("MongoDB unavailable; using JSON memory fallback:", e)
        mongo_client = None
        mongo_collection = None

def load_memory() -> Dict[str, List[Dict[str, str]]]:
    if mongo_collection is not None:
        docs = mongo_collection.find({}, {"_id": 0})
        return {d["user_id"]: d.get("turns", []) for d in docs}
    if os.path.exists(MEMORY_PATH):
        with open(MEMORY_PATH, "r") as f:
            return json.load(f)
    return {}

def save_memory(memory: Dict[str, List[Dict[str, str]]]):
    if mongo_collection is not None:
        for user_id, turns in memory.items():
            mongo_collection.replace_one(
                {"user_id": user_id}, {"user_id": user_id, "turns": turns}, upsert=True
            )
        return
    with open(MEMORY_PATH, "w") as f:
        json.dump(memory, f, indent=2)

memory_store = load_memory()

def add_memory(user_id: str, role: str, content: str):
    memory_store.setdefault(user_id, [])
    memory_store[user_id].append({"role": role, "content": content, "ts": time.time()})
    # Keep the store compact while retaining long-term history.
    memory_store[user_id] = memory_store[user_id][-100:]
    save_memory(memory_store)

def get_recent_memory(user_id: str, n: int = 8) -> str:
    turns = memory_store.get(user_id, [])[-n:]
    return "\n".join(f"{t['role']}: {t['content']}" for t in turns)

def summarize_user_preferences(user_id: str) -> str:
    history = "\n".join(
        f"{t['role']}: {t['content']}" for t in memory_store.get(user_id, [])
    )
    if not history:
        return "No known preferences yet."
    prompt = (
        "From this conversation history, extract durable user facts/preferences "
        "(likes, role, goals, constraints). Return a short bullet list.\n\n" + history
    )
    return llm.invoke(prompt).content

print("Memory backend:", "MongoDB" if mongo_collection is not None else "JSON fallback")


## 6. Dynamic Tool Integration

**Step 6 of the methodology:** integrate APIs / live data so the bot can answer questions that
static knowledge (scraped pages) can't — anything time-sensitive or not in the corpus.

Uses DuckDuckGo search (free, no API key) as the live-data tool.


In [ ]:
from duckduckgo_search import DDGS

def web_search_tool(query: str, max_results: int = 4) -> str:
    """Fetches live web results for time-sensitive / out-of-corpus questions."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No live results found."
        return "\n\n".join(f"{r['title']}: {r['body']} ({r['href']})" for r in results)
    except Exception as e:
        return f"Tool error: {e}"

# quick smoke test
print(web_search_tool("current weather in Bhubaneswar")[:500])


## 7. LangGraph Workflow — Orchestration Layer

**Step 5 of the methodology.** This is the "Dynamic Intelligence Layer" from the system
overview: a router node decides whether a query needs the **RAG** node, the **Tool** node, or
can be answered directly from **Memory**, then a final **Model** node composes the reply using
whatever context was gathered.

Graph: `router → {rag_node | tool_node | memory_node} → responder`


In [ ]:
from langgraph.graph import StateGraph, END

class ChatState(TypedDict):
    user_id: str
    question: str
    route: str
    rag_result: Optional[Dict[str, Any]]
    tool_result: Optional[str]
    memory_context: str
    final_answer: str

ROUTER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Classify the user question into exactly one category:\n"
     "- 'hybrid' — knowledge question where static semantic context and graph relationships are useful\n"
     "- 'tool' — needs current/live information (latest, today, weather, prices, news)\n"
     "- 'memory' — asks about the user or prior conversation\n"
     "Return ONLY one word: hybrid, tool, or memory."),
    ("human", "{question}"),
])

def router_node(state: ChatState) -> ChatState:
    route = (ROUTER_PROMPT | llm).invoke({"question": state["question"]}).content.strip().lower()
    if route not in {"hybrid", "tool", "memory"}:
        route = "hybrid"
    state["route"] = route
    return state

def hybrid_node(state: ChatState) -> ChatState:
    state["rag_result"] = rag_answer(state["question"])
    return state

def tool_node(state: ChatState) -> ChatState:
    state["tool_result"] = web_search_tool(state["question"])
    return state

def memory_node(state: ChatState) -> ChatState:
    state["memory_context"] = get_recent_memory(state["user_id"])
    return state

RESPONDER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a personalized, memory-aware assistant. Use the provided evidence to answer. "
     "For hybrid questions, use both semantic context and graph facts. For live questions, use "
     "the tool results. For memory questions, use the user's stored conversation context. "
     "Do not mention internal routing."),
    ("human",
     "Recent memory:\n{memory_context}\n\n"
     "Hybrid retrieval result:\n{rag_context}\n\n"
     "Live tool result:\n{tool_context}\n\n"
     "Question: {question}"),
])

def responder_node(state: ChatState) -> ChatState:
    rag_context = "N/A"
    if state.get("rag_result"):
        r = state["rag_result"]
        rag_context = f"Answer draft: {r['answer']}\nGraph facts: {r['kg_facts']}\nSources: {r['sources']}"
    response = (RESPONDER_PROMPT | llm).invoke({
        "memory_context": state.get("memory_context") or get_recent_memory(state["user_id"]),
        "rag_context": rag_context,
        "tool_context": state.get("tool_result") or "N/A",
        "question": state["question"],
    })
    state["final_answer"] = response.content
    return state

def route_decision(state: ChatState) -> str:
    return state["route"]

workflow = StateGraph(ChatState)
workflow.add_node("router", router_node)
workflow.add_node("hybrid", hybrid_node)
workflow.add_node("tool", tool_node)
workflow.add_node("memory", memory_node)
workflow.add_node("responder", responder_node)
workflow.set_entry_point("router")
workflow.add_conditional_edges("router", route_decision, {
    "hybrid": "hybrid", "tool": "tool", "memory": "memory",
})
workflow.add_edge("hybrid", "responder")
workflow.add_edge("tool", "responder")
workflow.add_edge("memory", "responder")
workflow.add_edge("responder", END)

chatbot_graph = workflow.compile()
print("LangGraph workflow compiled.")


In [ ]:
# --- Visualize the graph structure ---
from IPython.display import Image, display
try:
    display(Image(chatbot_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Mermaid rendering unavailable in this environment:", e)
    print(chatbot_graph.get_graph().draw_ascii())


## 8. Chat Function — End-to-End System

Each turn passes through LangGraph, then the interaction is persisted to the memory backend.
The hybrid path combines vector retrieval and graph facts before answer generation.


In [ ]:
def chat(user_id: str, question: str, verbose: bool = True) -> str:
    state: ChatState = {
        "user_id": user_id,
        "question": question,
        "route": "",
        "rag_result": None,
        "tool_result": None,
        "memory_context": get_recent_memory(user_id),
        "final_answer": "",
    }
    result = chatbot_graph.invoke(state)

    add_memory(user_id, "user", question)
    add_memory(user_id, "assistant", result["final_answer"])

    if verbose:
        print(f"[routed to: {result['route']}]")
    return result["final_answer"]

# --- Demo conversation ---
user = "demo_user"
print("Bot:", chat(user, "Hi, my name is Alex and I'm interested in AI systems."))
print()
print("Bot:", chat(user, "What is retrieval-augmented generation?"))
print()
print("Bot:", chat(user, "What's the latest news about Gemini?"))
print()
print("Bot:", chat(user, "What's my name and what am I interested in?"))


In [ ]:
# FastAPI serving layer -----------------------------------------------------
# The app is defined here and can be launched with: uvicorn <module>:app --reload
# In Colab, call start_api() only if you want to expose the endpoint.

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Memory-Augmented Hybrid RAG Chatbot", version="1.0")

class ChatRequest(BaseModel):
    user_id: str
    question: str

class ChatResponse(BaseModel):
    user_id: str
    question: str
    answer: str

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/chat", response_model=ChatResponse)
def chat_endpoint(request: ChatRequest):
    answer = chat(request.user_id, request.question, verbose=False)
    return ChatResponse(user_id=request.user_id, question=request.question, answer=answer)

print("FastAPI app ready: GET /health, POST /chat")


## 9. Evaluation Framework

**Step 7 of the methodology:** evaluate the hybrid RAG response using an **LLM-as-judge** approach.

The evaluation measures the three dimensions explicitly required by the project specification:
- **Context relevance**
- **Faithfulness**
- **Answer correctness**

The test set is intentionally small for notebook demonstration; expand it for a formal benchmark.


In [ ]:
EVAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an evaluation judge for a RAG system. Score the following on a 1-5 scale each:\n"
     "- context_relevance: how relevant the retrieved context is to the question\n"
     "- faithfulness: how well the answer is grounded in the context (no hallucination)\n"
     "- answer_correctness: how well the answer addresses the question\n"
     "Return ONLY valid JSON: {{\"context_relevance\": int, \"faithfulness\": int, "
     "\"answer_correctness\": int, \"justification\": str}}"),
    ("human",
     "Question: {question}\n\nContext:\n{context}\n\nAnswer:\n{answer}"),
])

def evaluate_response(question: str, context: str, answer: str) -> Dict[str, Any]:
    raw = (EVAL_PROMPT | llm).invoke({
        "question": question, "context": context, "answer": answer,
    }).content
    try:
        cleaned = raw.strip().strip("```").replace("json", "", 1).strip()
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return {"raw_response": raw}

# --- Run evaluation over a small test set ---
test_questions = [
    "What is retrieval-augmented generation?",
    "What is a knowledge graph used for?",
    "How do chatbots work?",
]

eval_results = []
for q in test_questions:
    result = rag_answer(q)
    score = evaluate_response(q, result["context"], result["answer"])
    eval_results.append({"question": q, **score})
    print(f"Q: {q}\n  -> {score}\n")

import pandas as pd
eval_df = pd.DataFrame(eval_results)
eval_df


In [ ]:
# Aggregate scores
numeric_cols = [c for c in ["context_relevance", "faithfulness", "answer_correctness"] if c in eval_df.columns]
if numeric_cols:
    print("Average scores across test set:")
    print(eval_df[numeric_cols].mean())
